# Time Series Forecasting Models -- ARIMA, LSTM, & Facebook Prophet

In [7]:
## Import Libraries
import pandas as pd
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from prophet import Prophet
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout


1.ARIMA Model - Forecasting

In [ ]:
### Load and Prepare Data
df = pd.read_csv(r"your Cleaned_bit_usd_data.csv", header=[0,1])

### Fix Multi_Index Columns
df.columns = df.columns.get_level_values(0)

## Convert Date
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

df = df.asfreq('D')
df['Close'] = df['Close'].fillna(method='ffill')

## Keep Only Close Price 
df = df[['Close']].dropna()
df.head()

C:\Users\anand\AppData\Local\Temp\ipykernel_14296\3032156299.py:12: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['Close'] = df['Close'].fillna(method='ffill')


,Close
Date,
2014-09-17,457.334015
2014-09-18,424.440002
2014-09-19,394.795990
2014-09-20,408.903992
2014-09-21,398.821014


In [6]:
#### Train the ARIMA Model
arima_model = ARIMA(df['Close'], order=(5, 1, 0))
arima_fit = arima_model.fit()

## Forecast
arima_forecast = arima_fit.forecast(steps=30)
arima_forecast

2026-03-30    66805.004038
2026-03-31    66731.862828
2026-04-01    66688.560594
2026-04-02    66696.704745
2026-04-03    66705.069486
2026-04-04    66703.640726
2026-04-05    66701.834222
2026-04-06    66701.195659
2026-04-07    66701.479085
2026-04-08    66701.606120
2026-04-09    66701.557152
2026-04-10    66701.517858
2026-04-11    66701.510986
2026-04-12    66701.518133
2026-04-13    66701.519673
2026-04-14    66701.518304
2026-04-15    66701.517542
2026-04-16    66701.517532
2026-04-17    66701.517685
2026-04-18    66701.517693
2026-04-19    66701.517659
2026-04-20    66701.517646
2026-04-21    66701.517649
2026-04-22    66701.517651
2026-04-23    66701.517651
2026-04-24    66701.517650
2026-04-25    66701.517650
2026-04-26    66701.517650
2026-04-27    66701.517650
2026-04-28    66701.517650
Freq: D, Name: predicted_mean, dtype: float64

In [ ]:
#### To Save the ARIMA Model
base_path = r"your path Models"
os.makedirs(base_path, exist_ok=True)

file_path = os.path.join(base_path, "arima_model.pkl")

with open(file_path, 'wb') as f:
    pickle.dump(arima_fit, f)

print(os.listdir(base_path))

['arima_model.pkl']


  # 2. LSTM Model Forecasting ---

In [10]:
### Scale Data
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[['Close']])

### Create Sequence
def create_data(data, step=60):
    X, y = [], []
    for i in range(len(data)-step):
        X.append(data[i:i+step])
        y.append(data[i+step])
    return np.array(X), np.array(y)

X, y = create_data(scaled_data)
X, y

(array([[[0.00224148],
         [0.00197743],
         [0.00173947],
         ...,
         [0.00194769],
         [0.00176372],
         [0.00158965]],
 
        [[0.00197743],
         [0.00173947],
         [0.00185272],
         ...,
         [0.00176372],
         [0.00158965],
         [0.00168397]],
 
        [[0.00173947],
         [0.00185272],
         [0.00177178],
         ...,
         [0.00158965],
         [0.00168397],
         [0.00168016]],
 
        ...,
 
        [[0.70711974],
         [0.71382601],
         [0.71448425],
         ...,
         [0.56464041],
         [0.57099824],
         [0.55078336]],
 
        [[0.71382601],
         [0.71448425],
         [0.67737403],
         ...,
         [0.57099824],
         [0.55078336],
         [0.53109031]],
 
        [[0.71448425],
         [0.67737403],
         [0.67389876],
         ...,
         [0.55078336],
         [0.53109031],
         [0.53109031]]], shape=(4152, 60, 1)),
 array([[0.00168397],
        [0.0

In [11]:
# Split
split = int(len(X)*0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Model
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(60,1)),
    LSTM(50),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.fit(X_train, y_train, epochs=5, batch_size=32)
model

c:\Users\anand\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - loss: 0.0038
Epoch 2/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 2.4841e-04
Epoch 3/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 2.0613e-04
Epoch 4/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 2.0383e-04
Epoch 5/5
104/104 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 1.5572e-04


<Sequential name=sequential, built=True>

In [ ]:
#### To Save the LSTM Model
base_path = r"your path Models"
os.makedirs(base_path, exist_ok=True)

file_path = os.path.join(base_path, "lstm_model.h5")
model.save(file_path)

## Save Scaler
scaler_path = os.path.join(base_path, "scaler_model.pkl")

with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print(os.listdir(base_path))

['arima_model.pkl', 'lstm_model.h5', 'scaler_model.pkl']


 # 3. Prophet Forecast

In [16]:
# Prepare data
prophet_df = df.reset_index()
prophet_df.columns = ['ds', 'y']

# Train model
prophet_model = Prophet()
prophet_model.fit(prophet_df)
prophet_model

DEBUG:cmdstanpy:cmd: where.exe tbb.dll
cwd: None
DEBUG:cmdstanpy:Adding TBB (c:\Users\anand\AppData\Local\Programs\Python\Python313\Lib\site-packages\prophet\stan_model\cmdstan-2.33.1\stan\lib\stan_math\lib\tbb) to PATH
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
DEBUG:cmdstanpy:input tempfile: C:\Users\anand\AppData\Local\Temp\tmp7sho6scw\w496f5jf.json
DEBUG:cmdstanpy:input tempfile: C:\Users\anand\AppData\Local\Temp\tmp7sho6scw\_zji7d8w.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['C:\\Users\\anand\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages\\prophet\\stan_model\\prophet_model.bin', 'random', 'seed=28093', 'data', 'file=C:\\Users\\anand\\AppData\\Local\\Temp\\tmp7sho6scw\\w496f5jf.json', 'init=C:\\Users\\anand\\AppData\\Local\\Temp\\tmp7sho6scw\\_zji7d8w.json', 'output', 'file=C:\\Users\\anand\\AppData\\Local\\Temp\\tmp7sho6scw\\prophet_modelirs

In [17]:
# Forecast future
future = prophet_model.make_future_dataframe(periods=30)
forecast = prophet_model.predict(future)
forecast

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,additive_terms,additive_terms_lower,additive_terms_upper,weekly,weekly_lower,weekly_upper,yearly,yearly_lower,yearly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat
0,2014-09-17,-175.093920,-13537.453498,9231.453283,-175.093920,-175.093920,-1773.343941,-1773.343941,-1773.343941,106.363670,106.363670,106.363670,-1879.707611,-1879.707611,-1879.707611,0.0,0.0,0.0,-1948.437861
1,2014-09-18,-173.674704,-12604.281253,9392.989244,-173.674704,-173.674704,-1884.504360,-1884.504360,-1884.504360,-29.221908,-29.221908,-29.221908,-1855.282452,-1855.282452,-1855.282452,0.0,0.0,0.0,-2058.179064
2,2014-09-19,-172.255488,-12809.125604,8886.417199,-172.255488,-172.255488,-1862.038722,-1862.038722,-1862.038722,-36.344015,-36.344015,-36.344015,-1825.694707,-1825.694707,-1825.694707,0.0,0.0,0.0,-2034.294210
3,2014-09-20,-170.836271,-13219.343737,9459.269425,-170.836271,-170.836271,-1850.725009,-1850.725009,-1850.725009,-58.908635,-58.908635,-58.908635,-1791.816373,-1791.816373,-1791.816373,0.0,0.0,0.0,-2021.561280
4,2014-09-21,-169.417055,-13131.197335,9689.922488,-169.417055,-169.417055,-1812.814326,-1812.814326,-1812.814326,-58.274612,-58.274612,-58.274612,-1754.539715,-1754.539715,-1754.539715,0.0,0.0,0.0,-1982.231381
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4237,2026-04-24,113852.521404,105764.052727,126632.034626,113852.521404,113852.521404,2541.052504,2541.052504,2541.052504,-36.344015,-36.344015,-36.344015,2577.396519,2577.396519,2577.396519,0.0,0.0,0.0,116393.573908
4238,2026-04-25,113927.640654,104919.959151,127190.800918,113927.640654,113927.640654,2488.035056,2488.035056,2488.035056,-58.908635,-58.908635,-58.908635,2546.943692,2546.943692,2546.943692,0.0,0.0,0.0,116415.675710
4239,2026-04-26,114002.759905,105030.817203,126964.403891,114002.759905,114002.759905,2461.062424,2461.062424,2461.062424,-58.274612,-58.274612,-58.274612,2519.337036,2519.337036,2519.337036,0.0,0.0,0.0,116463.822329
4240,2026-04-27,114077.879155,105559.579903,127363.265620,114077.879155,114077.879155,2565.217626,2565.217626,2565.217626,70.644736,70.644736,70.644736,2494.572890,2494.572890,2494.572890,0.0,0.0,0.0,116643.096781


In [ ]:
#### To Save the Prophet Model
base_path = r"your path Models"
os.makedirs(base_path, exist_ok=True)

file_path = os.path.join(base_path, "prophet_model.pkl")

with open(file_path, 'wb') as f:
    pickle.dump(prophet_model, f)

print(os.listdir(base_path))

['arima_model.pkl', 'lstm_model.h5', 'prophet_model.pkl', 'scaler_model.pkl']
